# Distribution of User Comments in January 2020 (Meta/Facebook)

Write a query to calculate the distribution of comments by the count of users that joined Meta/Facebook between 2018 and 2020, for the month of January 2020.

The output should contain a count of comments and the corresponding number of users that made that number of comments in Jan-2020. For example, you'll be counting how many users made 1 comment, 2 comments, 3 comments, 4 comments, etc in Jan-2020. Your left column in the output will be the number of comments while your right column in the output will be the number of users. Sort the output from the least number of comments to highest.

To add some complexity, there might be a bug where an user post is dated before the user join date. You'll want to remove these posts from the result.

🌀By solving this, you'll learn how to use Mutiple Cte, join, group by. Give it a try and share the output! 👇

In [0]:
%skip 
DROP TABLE  ska_catalog2.bronze.fb_users

In [0]:
%skip
CREATE TABLE ska_catalog2.bronze.fb_users (
  city_id BIGINT,
  device BIGINT,
  id BIGINT,
  joined_at DATE,
  name VARCHAR(255)
);

INSERT INTO ska_catalog2.bronze.fb_users (
  city_id, device, id, joined_at, name
) VALUES
  (101, 1, 1, '2019-06-15', 'Alice'),
  (102, 2, 2, '2020-03-10', 'Bob'),
  (103, 1, 3, '2018-11-25', 'Charlie'),
  (104, 3, 4, '2017-09-05', 'David'),
  (105, 1, 5, '2019-01-20', 'Eve'),
  (106, 2, 6, '2020-01-05', 'Frank');

CREATE TABLE ska_catalog2.bronze.fb_comments (
  body VARCHAR(100),
  created_at TIMESTAMP,
  user_id BIGINT
);

INSERT INTO ska_catalog2.bronze.fb_comments (
  body, created_at, user_id
) VALUES
  ('Great post!', '2020-01-01 10:00:00', 1),
  ('Interesting article', '2020-01-02 12:30:00', 1),
  ('Thanks for sharing!', '2020-01-05 08:20:00', 2),
  ('Nice update', '2020-01-08 15:45:00', 3),
  ('Good job', '2020-01-12 14:00:00', 3),
  ('Helpful content', '2020-01-14 09:00:00', 3),
  ('Loved it!', '2020-01-18 11:10:00', 5),
  ('Noted', '2020-01-20 17:40:00', 6),
  ('Cool!', '2020-01-22 08:55:00', 6),
  ('Agreed', '2020-01-25 19:30:00', 6),
  ('Well written', '2020-01-28 20:45:00', 1),
  ('Informative', '2020-01-30 13:50:00', 5),
  ('Awesome', '2019-12-31 23:59:00', 2);

In [0]:
SELECT * FROM ska_catalog2.bronze.fb_users

In [0]:
SELECT * FROM ska_catalog2.bronze.fb_comments

In [0]:
WITH ValidUsers AS (
  SELECT 
    id AS `user_id`,
    joined_at
  FROM
    ska_catalog2.bronze.fb_users
  WHERE  YEAR(joined_at) BETWEEN 2018 AND 2020
),
ValidCommnets AS (
  SELECT 
    fc.user_id,
    COUNT(fc.body) AS `comment_count`
  FROM
    ska_catalog2.bronze.fb_comments fc
  INNER JOIN ValidUsers vu
  ON fc.user_id = vu.user_id
  WHERE 
    fc.created_at >= '2020-01-01'
    AND fc.created_at < '2020-02-01'
    AND fc.created_at >= vu.joined_at
  GROUP BY
    fc.user_id
),
CommonDistribution AS (
  SELECT 
    comment_count,
    COUNT(user_id) AS user_count
  FROM
    ValidCommnets
  GROUP BY 
    comment_count
)
SELECT
  comment_count,
  user_count
FROM
   CommonDistribution
ORDER BY comment_count ASC;